In [1]:
import os
import re
import time
import json
import random
import traceback
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm

In [2]:
# =========================================================
# CONFIGURACIÓN GENERAL
# =========================================================

OUTPUT_DIR = "scraping_output_round2"
RAW_DIR = os.path.join(OUTPUT_DIR, "raw")
FINAL_DIR = os.path.join(OUTPUT_DIR, "final")
LOG_DIR = os.path.join(OUTPUT_DIR, "logs")

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(FINAL_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 14_3) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/134.0.0.0 Safari/537.36",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/133.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:136.0) Gecko/20100101 Firefox/136.0",
]

DEFAULT_TIMEOUT = 20
DEFAULT_RETRIES = 3

SCRAPER_CONFIG = {
    "pincali": {"delay": (2, 5), "retries": 3},
    "icasas": {"delay": (2, 5), "retries": 3},
    "yapo": {"delay": (2, 6), "retries": 3},
    "properati": {"delay": (2, 5), "retries": 3},
    "encuentra24": {"delay": (2, 6), "retries": 3},
    "quintoandar": {"delay": (2, 5), "retries": 3},
}

In [8]:
# =========================================================
# HELPERS
# =========================================================

def build_headers():
    return {
        "User-Agent": random.choice(USER_AGENTS),
        "Accept-Language": "es-MX,es;q=0.9,en;q=0.8",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
        "Connection": "keep-alive",
        "Upgrade-Insecure-Requests": "1"
    }


def random_sleep(min_sec=2, max_sec=5):
    time.sleep(random.uniform(min_sec, max_sec))


def safe_request(url, session=None, scraper_name="generic", timeout=DEFAULT_TIMEOUT):
    cfg = SCRAPER_CONFIG.get(scraper_name, {"delay": (2, 5), "retries": DEFAULT_RETRIES})
    retries = cfg["retries"]
    delay_min, delay_max = cfg["delay"]

    client = session if session is not None else requests.Session()

    for attempt in range(1, retries + 1):
        try:
            response = client.get(url, headers=build_headers(), timeout=timeout)

            if response.status_code == 200:
                random_sleep(delay_min, delay_max)
                return response

            wait_time = (2 ** attempt) + random.uniform(0.5, 1.5)
            print(f"[{scraper_name}] status {response.status_code} en {url}. Reintento {attempt}/{retries} en {wait_time:.1f}s")
            time.sleep(wait_time)

        except Exception as e:
            wait_time = (2 ** attempt) + random.uniform(0.5, 1.5)
            print(f"[{scraper_name}] error en request: {e}. Reintento {attempt}/{retries} en {wait_time:.1f}s")
            time.sleep(wait_time)

    return None


def save_csv(df, filename, folder):
    path = os.path.join(folder, filename)
    df.to_csv(path, index=False, encoding="utf-8-sig")
    return path


def save_log(log_rows, filename="scraping_log.csv"):
    if not log_rows:
        return None
    log_df = pd.DataFrame(log_rows)
    path = os.path.join(LOG_DIR, filename)
    log_df.to_csv(path, index=False, encoding="utf-8-sig")
    return path


def standardize_columns(df):
    expected = [
        "name", "price", "currency", "type", "size", "bedrooms", "bathrooms",
        "latitude", "longitude", "street", "region", "locality",
        "consultation_date", "source", "country", "operation"
    ]
    for col in expected:
        if col not in df.columns:
            df[col] = None
    return df[expected]


# =========================================================
# SCRAPERS
# =========================================================

def scrape_pincali(operation="venta", tipo_inmueble="casas", max_pages=100):
    scraper_name = "pincali"
    consultation_date = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    operation_type = "sell" if operation == "venta" else "rent"
    prop_type = "House" if tipo_inmueble == "casas" else "Apartment"

    all_tables = []
    session = requests.Session()

    for i in tqdm(range(1, max_pages + 1), desc=f"Scraping Pincali - {tipo_inmueble} - {operation_type}"):
        url = f"https://www.pincali.com/inmuebles/{tipo_inmueble}-en-{operation}?page={i}"
        response = safe_request(url, session=session, scraper_name=scraper_name)

        if response is None:
            print(f"[pincali] No se pudo descargar página {i}")
            continue

        soup = BeautifulSoup(response.text, "html.parser")
        basic_data = soup.find_all("div", class_="property__component")

        if not basic_data:
            print(f"[pincali] Sin resultados en página {i}. Se detiene.")
            break

        rows = []

        for elementos in basic_data:
            price_tag = elementos.find("li", class_="price")
            name_tag = elementos.find("div", class_="title")
            location_tag = elementos.find("div", class_="location")

            lat = elementos.get("data-lat")
            lng = elementos.get("data-long")

            street = None
            region = None
            locality = None

            if location_tag:
                location_text = location_tag.get_text(strip=True)
                partes = [p.strip() for p in location_text.split(",")]
                street = location_text
                locality = partes[-1] if len(partes) > 0 else None
                region = partes[-2] if len(partes) > 1 else None

            rec = None
            bath = None
            sup = None

            features = elementos.find("div", class_="features")
            if features:
                for item in features.find_all("div"):
                    texto = item.get_text(strip=True).lower()
                    numero = re.search(r"\d+", texto)
                    numero = int(numero.group()) if numero else None

                    if "rec" in texto:
                        rec = numero
                    elif "bañ" in texto:
                        bath = numero
                    elif "m²" in texto or "m2" in texto:
                        sup = numero

            rows.append({
                "name": name_tag.get_text(strip=True) if name_tag else None,
                "price": price_tag.get_text(strip=True) if price_tag else None,
                "type": prop_type,
                "size": sup,
                "bedrooms": rec,
                "bathrooms": bath,
                "latitude": float(lat) if lat else None,
                "longitude": float(lng) if lng else None,
                "street": street,
                "region": region,
                "locality": locality,
                "consultation_date": consultation_date,
                "source": "Pincali",
                "country": "Mexico",
                "operation": operation_type,
                "currency": None
            })

        page_df = pd.DataFrame(rows)
        all_tables.append(page_df)

    if not all_tables:
        return pd.DataFrame()

    df = pd.concat(all_tables, ignore_index=True)
    return standardize_columns(df)


def scrape_icasas(estado, operation="venta", tipo_inmueble="casas", max_pages=100):
    scraper_name = "icasas"
    if operation == "venta":
        base_url = "https://www.icasas.mx/venta/habitacionales-{}-{}-2_5_1_0_0_0/list/p_{}"
    elif operation == "renta":
        base_url = "https://www.icasas.mx/renta/habitacionales-{}-{}-2_5_1_0_0_0/list/p_{}"
    else:
        raise ValueError("Invalid operation type. Use 'venta' or 'renta'.")

    all_data = []
    session = requests.Session()
    consultation_date = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    for pagina in tqdm(range(1, max_pages + 1), desc=f"Scraping iCasas - {operation}, {tipo_inmueble} in {estado}"):
        url = base_url.format(tipo_inmueble, estado, pagina)
        response = safe_request(url, session=session, scraper_name=scraper_name)

        if response is None:
            print(f"[icasas] No se pudo descargar página {pagina}")
            continue

        soup = BeautifulSoup(response.text, "html.parser")
        resultados = soup.find_all("li", class_="serp-snippet ad featured")

        if not resultados:
            print(f"[icasas] Sin resultados en página {pagina}. Se detiene.")
            break

        for item in resultados:
            a_tag = item.find("a", class_="detail-redirection")
            lat_tag = item.find("meta", itemprop="latitude")
            lon_tag = item.find("meta", itemprop="longitude")

            all_data.append({
                "name": a_tag.get_text(strip=True) if a_tag else None,
                "price": item.find("div", class_="price").get_text(strip=True) if item.find("div", class_="price") else None,
                "size": item.find("span", class_="areaBuilt").get_text(strip=True) if item.find("span", class_="areaBuilt") else None,
                "bedrooms": item.find("span", class_="rooms").get_text(strip=True) if item.find("span", class_="rooms") else None,
                "bathrooms": item.find("span", class_="bathrooms").get_text(strip=True) if item.find("span", class_="bathrooms") else None,
                "latitude": lat_tag["content"] if lat_tag else None,
                "longitude": lon_tag["content"] if lon_tag else None,
                "street": None,
                "region": estado,
                "locality": None,
                "consultation_date": consultation_date,
                "source": "iCasas",
                "country": "Mexico",
                "operation": "sell" if operation == "venta" else "rent",
                "type": "House" if tipo_inmueble == "casas" else "Apartment",
                "currency": None
            })

    df = pd.DataFrame(all_data)
    return standardize_columns(df)


def scrape_yapo(operation="venta", tipo_inmueble="apartamentos", max_pages=100):
    scraper_name = "yapo"
    data = []
    session = requests.Session()
    consultation_date = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    for i in tqdm(range(1, max_pages + 1), desc=f"Scraping Yapo - {operation} - {tipo_inmueble}"):

        if operation == "venta":
            url = f"https://www.yapo.cl/searchresult/bienes-raices-venta-de-propiedades-{tipo_inmueble}.{i}"
        elif operation in ["alquiler", "arriendo"]:
            url = f"https://www.yapo.cl/searchresult/bienes-raices-alquiler-{tipo_inmueble}.{i}"
        else:
            raise ValueError("operation debe ser 'venta' o 'alquiler/arriendo'")

        response = safe_request(url, session=session, scraper_name=scraper_name)

        if response is None:
            print(f"[yapo] No se pudo descargar página {i}")
            continue

        soup = BeautifulSoup(response.text, "html.parser")
        ads = soup.find_all("div", class_="d3-ad-tile__content")

        if not ads:
            print(f"[yapo] Sin resultados en página {i}. Se detiene.")
            break

        for ad in ads:
            details = ad.find_all("li", class_="d3-ad-tile__details-item")

            bedrooms = None
            bathrooms = None
            size = None

            for li in details:
                icon = li.find("use")
                if icon:
                    href = icon.get("xlink:href", "")
                    if "#bed" in href:
                        bedrooms = li.get_text(strip=True)
                    elif "#bath" in href:
                        bathrooms = li.get_text(strip=True)
                    elif "#resize" in href:
                        size = li.get_text(strip=True)

            data.append({
                "consultation_date": consultation_date,
                "source": "Yapo",
                "type": "Apartment" if tipo_inmueble == "apartamentos" else "House",
                "operation": "sell" if operation == "venta" else "rent",
                "locality": ad.find("div", class_="d3-ad-tile__location").get_text(strip=True) if ad.find("div", class_="d3-ad-tile__location") else None,
                "country": "Chile",
                "name": ad.find("span", class_="d3-ad-tile__title").get_text(strip=True) if ad.find("span", class_="d3-ad-tile__title") else None,
                "price": ad.find("div", class_="d3-ad-tile__price").get_text(strip=True) if ad.find("div", class_="d3-ad-tile__price") else None,
                "bedrooms": bedrooms,
                "bathrooms": bathrooms,
                "size": size,
                "latitude": None,
                "longitude": None,
                "street": None,
                "region": None,
                "currency": None
            })

    df = pd.DataFrame(data)
    return standardize_columns(df)


def scrape_properati(pais, operation="alquiler", max_pages=100):
    scraper_name = "properati"

    country_domain_map = {
        "Argentina": "ar",
        "Ecuador": "ec",
        "Colombia": "co",
        "Peru": "pe"
    }

    if pais not in country_domain_map:
        raise ValueError(f"Country must be one of {list(country_domain_map.keys())}")

    domain = country_domain_map[pais]
    consultation_date = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    session = requests.Session()
    all_pages = []

    for page in tqdm(range(1, max_pages + 1), desc=f"Scraping Properati - {pais} - {operation}"):
        url = f"https://www.properati.com.{domain}/s/{operation}/{page}"
        response = safe_request(url, session=session, scraper_name=scraper_name)

        if response is None:
            print(f"[properati] No se pudo descargar página {page}")
            continue

        soup = BeautifulSoup(response.text, "html.parser")

        basics = soup.find_all("div", class_="snippet__content")
        if len(basics) == 0:
            print(f"[properati] Sin resultados en página {page}. Se detiene.")
            break

        price, surface = [], []
        for element in basics:
            price.append(
                element.find("div", class_="price").text.strip()
                if element.find("div", class_="price") else None
            )
            surface.append(
                element.find("span", class_="properties__area").text.strip()
                if element.find("span", class_="properties__area") else None
            )

        basics_df = pd.DataFrame({"price": price, "size": surface})

        if len(basics_df) > 2:
            basics_df = basics_df.drop([0, 1]).reset_index(drop=True)

        scripts = soup.find_all("script", type="application/ld+json")
        if not scripts:
            continue

        try:
            data_json = json.loads(scripts[0].text)[0]["about"]
        except Exception:
            continue

        rows = []
        for item in data_json:
            geo = item.get("geo", {})
            address = item.get("address", {})

            rows.append({
                "name": item.get("name"),
                "type": item.get("@type"),
                "street": address.get("streetAddress"),
                "region": address.get("addressRegion"),
                "locality": address.get("addressLocality"),
                "bedrooms": item.get("numberOfBedrooms"),
                "bathrooms": item.get("numberOfBathroomsTotal"),
                "latitude": geo.get("latitude"),
                "longitude": geo.get("longitude"),
            })

        table = pd.DataFrame(rows)
        final_df = pd.concat([table.reset_index(drop=True), basics_df.reset_index(drop=True)], axis=1)

        final_df["operation"] = "rent" if operation == "alquiler" else "sell"
        final_df["consultation_date"] = consultation_date
        final_df["country"] = pais
        final_df["source"] = "Properati"
        final_df["currency"] = None

        all_pages.append(final_df)

    if not all_pages:
        return pd.DataFrame()

    df = pd.concat(all_pages, ignore_index=True)
    return standardize_columns(df)


def scrape_encuentra24(tipo="venta", tipo_inmueble="apartamentos", pais="panama", max_pages=100):
    scraper_name = "encuentra24"
    data = []
    session = requests.Session()
    consultation_date = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    for i in tqdm(range(1, max_pages + 1), desc=f"Scraping Encuentra24 - {pais} - {tipo} - {tipo_inmueble}"):
        if tipo == "venta":
            url = f"https://www.encuentra24.com/{pais}-es/searchresult/bienes-raices-venta-de-propiedades.{i}?q=withcat.bienes-raices-venta-de-propiedades-{tipo_inmueble}"
        else:
            url = f"https://www.encuentra24.com/{pais}-es/searchresult/bienes-raices-alquiler-{tipo_inmueble}.{i}"

        response = safe_request(url, session=session, scraper_name=scraper_name)

        if response is None:
            print(f"[encuentra24] No se pudo descargar página {i}")
            continue

        soup = BeautifulSoup(response.text, "html.parser")
        ads = soup.select("[class*='card_content']")

        if not ads:
            print(f"[encuentra24] Sin resultados en página {i}. Se detiene.")
            break

        for ad in ads:
            bedrooms = None
            bathrooms = None
            size = None

            specs = ad.select("[class*='card_spec']")
            for spec in specs[1:]:
                text = spec.get_text(strip=True).lower()
                if "rec" in text or "habitac" in text or "cuarto" in text:
                    bedrooms = spec.get_text(strip=True)
                elif "baño" in text or "bath" in text:
                    bathrooms = spec.get_text(strip=True)
                elif "m²" in text or "m2" in text or "mt²" in text:
                    size = spec.get_text(strip=True)

            title_el    = ad.select_one("[class*='card_title']")
            price_el    = ad.select_one("[class*='card_price']")
            subtitle_el = ad.select_one("[class*='card_subtitle']")

            data.append({
                "consultation_date": consultation_date,
                "source": "Encuentra24",
                "type": "Apartment" if tipo_inmueble == "apartamentos" else "House",
                "operation": "sell" if tipo == "venta" else "rent",
                "locality": subtitle_el.get_text(strip=True) if subtitle_el else None,
                "country": pais.capitalize(),
                "name": title_el.get_text(strip=True) if title_el else None,
                "price": price_el.get_text(strip=True) if price_el else None,
                "bedrooms": bedrooms,
                "bathrooms": bathrooms,
                "size": size,
                "latitude": None,
                "longitude": None,
                "street": None,
                "region": None,
                "currency": None
            })

    df = pd.DataFrame(data)
    return standardize_columns(df)

In [4]:
# =========================================================
# QUINTOANDAR — scraper via POST /v2/search/list (offset-based)
# =========================================================

import math

_QA_TYPE_LABEL = {
    "Apartamento":         "Apartamento",
    "Casa":                "Casa",
    "StudioOuKitchenette": "Studio",
    "Cobertura":           "Cobertura",
    "Sobrado":             "Sobrado",
    "Flat":                "Flat",
}

_QA_CITY_META = {
    "sao-paulo-sp-brasil": {
        "coordinate": {"lat": -23.55052, "lng": -46.633309},
        "viewport": {
            "east": -46.57726472479766, "north": -23.420994351978546,
            "south": -23.680014200367943, "west": -46.689187942571095,
        },
    },
}

_QA_FIELDS = [
    "id", "rent", "totalCost", "salePrice", "iptuPlusCondominium",
    "area", "address", "regionName", "city", "type",
    "forRent", "forSale", "isPrimaryMarket", "bedrooms",
    "parkingSpaces", "suites", "neighbourhood", "bathrooms",
    "isFurnished", "shortRentDescription", "shortSaleDescription",
]

_QA_AB_TEST = (
    "ab_beakman_search_services_anonymous_user_embedding_fallback:-1,"
    "ab_beakman_search_services_demand_concentration_v1_map_and_ssr_rent_experiment_v2:1,"
    "ab_beakman_search_services_demand_sufficiency_v1_sale_experiment:-1,"
    "ab_beakman_search_services_demand_sufficiency_v1_sale_experiment_rollout:false,"
    "ab_beakman_search_services_demand_sufficiency_v1_sale_experiment_rollout_v1:false,"
    "ab_beakman_search_services_demand_sufficiency_v1_sale_experiment_v1:-1,"
    "ab_beakman_search_services_feed_filter_search_profile_experiment:0,"
    "ab_beakman_search_services_feed_filter_search_profile_sale_experiment:0,"
    "ab_beakman_search_services_feed_filter_search_profile_sale_experiment_rollout:true,"
    "ab_beakman_search_services_hue_candidate_generation_experiment_v2:-1,"
    "ab_beakman_search_services_location_embedding_on_cg_experiment:1,"
    "ab_beakman_search_services_open_search_find:1,"
    "ab_beakman_search_services_open_search_migration:false,"
    "ab_beakman_search_services_open_search_migration_rollout:true"
)

_QA_API_URL = "https://apigw.prod.quintoandar.com.br/house-listing-search/v2/search/list"
_QA_PAGE_SIZE = 12


def _qa_get_session(city, operation):
    session = requests.Session()
    try:
        session.get(
            f"https://www.quintoandar.com.br/{operation}/imovel/{city}",
            headers=build_headers(), timeout=20
        )
    except Exception:
        pass
    return session


def _qa_build_body(city, operation, offset):
    meta = _QA_CITY_META.get(city, {
        "coordinate": {"lat": -23.55052, "lng": -46.633309},
        "viewport": {},
    })
    return {
        "slug": city,
        "topics": [],
        "fields": _QA_FIELDS,
        "sorting": {"criteria": "RELEVANCE", "order": "DESC"},
        "pagination": {"pageSize": _QA_PAGE_SIZE, "offset": offset},
        "context": {"listShowing": True, "mapShowing": False, "numPhotos": 4, "isSSR": False},
        "filters": {
            "enableFlexibleSearch": True,
            "businessContext": "RENT" if operation == "alugar" else "SALE",
            "location": {
                "coordinate": meta["coordinate"],
                "viewport": meta["viewport"],
                "neighborhoods": [],
                "countryCode": "BR",
            },
            "priceRange": [], "availability": "ANY", "occupancy": "ANY",
            "partnerIds": [], "specialConditions": [], "excludedSpecialConditions": [],
            "blocklist": [], "selectedHouses": [], "categories": [],
            "houseSpecs": {
                "area": {"range": {}}, "houseTypes": [], "amenities": [],
                "installations": [], "bathrooms": {"range": {}},
                "bedrooms": {"range": {}}, "parkingSpace": {"range": {}}, "suites": {"range": {}},
            },
        },
        "locationDescriptions": [{"description": city}],
    }


def _qa_api_headers(city, operation):
    return {
        **build_headers(),
        "Accept": "application/json",
        "Content-Type": "application/json",
        "Origin": "https://www.quintoandar.com.br",
        "Referer": f"https://www.quintoandar.com.br/{operation}/imovel/{city}",
        "x-ab-test": _QA_AB_TEST,
    }


def _qa_parse_response(data, operation, consultation_date):
    rows = []

    outer = data.get("hits", {}) if isinstance(data, dict) else {}
    if isinstance(outer, dict) and "hits" in outer:
        raw_hits = outer["hits"]
        total_raw = outer.get("total")
        total = total_raw.get("value") if isinstance(total_raw, dict) else total_raw
        items = [h.get("_source", h) for h in raw_hits if isinstance(h, dict)]
    elif isinstance(data, list):
        items, total = data, None
    else:
        return rows, None

    for h in items:
        if not isinstance(h, dict):
            continue

        price     = h.get("rent") if operation == "alugar" else h.get("salePrice")
        addr      = h.get("address") or {}
        street    = addr.get("address", "") if isinstance(addr, dict) else str(addr)
        city_name = (addr.get("city", "") if isinstance(addr, dict) else "") or h.get("city", "")
        raw_type  = h.get("type", "")
        bedrooms  = h.get("bedrooms")
        area      = h.get("area")
        neigh     = h.get("neighbourhood", "")

        label       = _QA_TYPE_LABEL.get(raw_type, raw_type)
        quartos_str = f"{bedrooms} quarto{'s' if bedrooms and bedrooms > 1 else ''}" if bedrooms else ""
        area_str    = f"{area}m²" if area else ""
        partes      = [p for p in [quartos_str, area_str] if p]
        location    = ", ".join([p for p in [neigh, city_name] if p])
        name        = f"{label} com {', '.join(partes)} em {location}" if partes else label

        prop_type = ("Apartment" if raw_type in ("Apartamento", "StudioOuKitchenette", "Flat")
                     else "House" if raw_type in ("Casa", "Sobrado", "Cobertura") else raw_type)

        rows.append({
            "name": name,
            "price": price,
            "currency": "BRL",
            "type": prop_type,
            "size": area,
            "bedrooms": bedrooms,
            "bathrooms": h.get("bathrooms"),
            "latitude": None,
            "longitude": None,
            "street": street,
            "region": city_name,
            "locality": neigh,
            "consultation_date": consultation_date,
            "source": "QuintoAndar",
            "country": "Brazil",
            "operation": "rent" if operation == "alugar" else "sell",
        })

    return rows, total


def scrape_quinto_andar(city, operation, max_pages=100):
    rows = []
    seen = set()
    total_listings = None

    session = _qa_get_session(city, operation)
    hdr = _qa_api_headers(city, operation)

    for page_num in tqdm(range(max_pages), desc=f"QuintoAndar {operation} {city}"):
        offset = page_num * _QA_PAGE_SIZE
        body   = _qa_build_body(city, operation, offset)

        response = session.post(_QA_API_URL, headers=hdr, json=body, timeout=20)
        random_sleep(2, 4)

        if response.status_code != 200:
            print(f"[quintoandar] Error {response.status_code} en offset {offset}: {response.text[:200]}")
            break

        try:
            data = response.json()
        except Exception as e:
            print(f"[quintoandar] Error JSON offset {offset}: {e}")
            break

        consultation_date = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        page_rows, total = _qa_parse_response(data, operation, consultation_date)

        if total_listings is None and total:
            total_listings = total
            print(f"[quintoandar] Total listings: {total_listings}")

        if not page_rows:
            print(f"[quintoandar] Sin resultados en offset {offset}. Se detiene.")
            break

        new_count = 0
        for item in page_rows:
            key = (item["name"], str(item["price"]))
            if key not in seen:
                seen.add(key)
                rows.append(item)
                new_count += 1

        if new_count == 0:
            print(f"[quintoandar] Offset {offset} sin filas nuevas. Se detiene.")
            break

        if total_listings and (offset + _QA_PAGE_SIZE) >= total_listings:
            print(f"[quintoandar] Fin: {offset + _QA_PAGE_SIZE}/{total_listings} listings.")
            break

    df = pd.DataFrame(rows)
    return standardize_columns(df)

In [5]:
# =========================================================
# TASKS — RONDA 2
# =========================================================

tasks = []

# =========================================================
# PROPERATI
# =========================================================
properati_countries = ["Argentina", "Ecuador", "Colombia", "Peru"]
properati_operations = ["venta", "alquiler"]

for country in properati_countries:
    for op in properati_operations:
        op_name = "sell" if op == "venta" else "rent"

        tasks.append({
            "name": f"properati_{country.lower()}_{op_name}",
            "func": scrape_properati,
            "kwargs": {
                "pais": country,
                "operation": op,
                "max_pages": 100
            }
        })


# =========================================================
# PINCALI — casas + departamentos
# =========================================================
for op in ["venta", "renta"]:
    op_name = "sell" if op == "venta" else "rent"

    for inmueble in ["casas", "departamentos"]:
        tasks.append({
            "name": f"pincali_mexico_{op_name}_{inmueble}",
            "func": scrape_pincali,
            "kwargs": {
                "operation": op,
                "tipo_inmueble": inmueble,
                "max_pages": 100
            }
        })


# =========================================================
# ICASAS — casas + departamentos
# =========================================================
for op in ["venta", "renta"]:
    op_name = "sell" if op == "venta" else "rent"

    for inmueble in ["casas", "departamentos"]:
        tasks.append({
            "name": f"icasas_mexico_{op_name}_{inmueble}",
            "func": scrape_icasas,
            "kwargs": {
                "estado": "distrito-federal",
                "operation": op,
                "tipo_inmueble": inmueble,
                "max_pages": 100
            }
        })


# =========================================================
# YAPO — apartamentos + casas
# =========================================================
for op in ["venta", "alquiler"]:
    op_name = "sell" if op == "venta" else "rent"

    for inmueble in ["apartamentos", "casas"]:
        tasks.append({
            "name": f"yapo_chile_{op_name}_{inmueble}",
            "func": scrape_yapo,
            "kwargs": {
                "operation": op,
                "tipo_inmueble": inmueble,
                "max_pages": 100
            }
        })


# =========================================================
# ENCUENTRA24 — apartamentos + casas
# =========================================================
encuentra24_countries = [
    "panama",
    "costa-rica",
    "guatemala",
    "el-salvador",
    "honduras",
    "nicaragua",
    "republica-dominicana"
]

for country in encuentra24_countries:
    for op in ["venta", "alquiler"]:
        op_name = "sell" if op == "venta" else "rent"

        for inmueble in ["apartamentos", "casas"]:
            tasks.append({
                "name": f"encuentra24_{country}_{op_name}_{inmueble}",
                "func": scrape_encuentra24,
                "kwargs": {
                    "tipo": op,
                    "tipo_inmueble": inmueble,
                    "pais": country,
                    "max_pages": 100
                }
            })


# =========================================================
# QUINTOANDAR
# =========================================================
for op in ["alugar", "comprar"]:
    op_name = "rent" if op == "alugar" else "sell"

    tasks.append({
        "name": f"quintoandar_brazil_{op_name}_sao_paulo",
        "func": scrape_quinto_andar,
        "kwargs": {
            "city": "sao-paulo-sp-brasil",
            "operation": op,
            "max_pages": 100
        }
    })

print(f"Total tasks: {len(tasks)}")
for t in tasks:
    print(f"  {t['name']}")

Total tasks: 50
  properati_argentina_sell
  properati_argentina_rent
  properati_ecuador_sell
  properati_ecuador_rent
  properati_colombia_sell
  properati_colombia_rent
  properati_peru_sell
  properati_peru_rent
  pincali_mexico_sell_casas
  pincali_mexico_sell_departamentos
  pincali_mexico_rent_casas
  pincali_mexico_rent_departamentos
  icasas_mexico_sell_casas
  icasas_mexico_sell_departamentos
  icasas_mexico_rent_casas
  icasas_mexico_rent_departamentos
  yapo_chile_sell_apartamentos
  yapo_chile_sell_casas
  yapo_chile_rent_apartamentos
  yapo_chile_rent_casas
  encuentra24_panama_sell_apartamentos
  encuentra24_panama_sell_casas
  encuentra24_panama_rent_apartamentos
  encuentra24_panama_rent_casas
  encuentra24_costa-rica_sell_apartamentos
  encuentra24_costa-rica_sell_casas
  encuentra24_costa-rica_rent_apartamentos
  encuentra24_costa-rica_rent_casas
  encuentra24_guatemala_sell_apartamentos
  encuentra24_guatemala_sell_casas
  encuentra24_guatemala_rent_apartamentos
  e

In [6]:
def run_task(task):
    """
    task = {
        "name": "properati_ar_sell",
        "func": scrape_properati,
        "kwargs": {"pais": "Argentina", "operation": "venta", "max_pages": 100}
    }
    """
    start = datetime.now()
    task_name = task["name"]

    try:
        print(f"[START] {task_name}")
        df = task["func"](**task["kwargs"])

        if df is None or df.empty:
            print(f"[EMPTY] {task_name}")
            return {
                "task_name": task_name,
                "status": "empty",
                "rows": 0,
                "file": None,
                "df": pd.DataFrame()
            }

        raw_file = f"{task_name}_raw.csv"
        raw_path = save_csv(df, raw_file, RAW_DIR)

        end = datetime.now()
        print(f"[OK] {task_name} | filas={len(df)}")

        return {
            "task_name": task_name,
            "status": "success",
            "rows": len(df),
            "file": raw_path,
            "started_at": start,
            "finished_at": end,
            "seconds": (end - start).total_seconds(),
            "df": df
        }

    except Exception as e:
        end = datetime.now()
        error_txt = traceback.format_exc()

        print(f"[ERROR] {task_name}: {e}")

        return {
            "task_name": task_name,
            "status": "error",
            "rows": 0,
            "file": None,
            "started_at": start,
            "finished_at": end,
            "seconds": (end - start).total_seconds(),
            "error": str(e),
            "traceback": error_txt,
            "df": pd.DataFrame()
        }


def run_scraping_pipeline(tasks, max_workers=4):

    results = []
    log_rows = []
    dfs = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_task = {executor.submit(run_task, task): task for task in tasks}

        for future in as_completed(future_to_task):
            result = future.result()
            results.append(result)

            log_rows.append({
                "task_name": result.get("task_name"),
                "status": result.get("status"),
                "rows": result.get("rows"),
                "file": result.get("file"),
                "started_at": result.get("started_at"),
                "finished_at": result.get("finished_at"),
                "seconds": result.get("seconds"),
                "error": result.get("error")
            })

            if result.get("status") == "success" and result.get("df") is not None and not result["df"].empty:
                dfs.append(result["df"])

    save_log(log_rows)

    if not dfs:
        print("No se obtuvieron datos.")
        return pd.DataFrame(), pd.DataFrame(log_rows)

    tabla_bronce = pd.concat(dfs, ignore_index=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    final_path = save_csv(tabla_bronce, f"bronze_full_{timestamp}.csv", RAW_DIR)

    print(f"Bronze data guardada en: {final_path}")

    return tabla_bronce, pd.DataFrame(log_rows)


def group_tasks(tasks):
    groups = {
        "properati": [],
        "encuentra24": [],
        "pincali": [],
        "icasas": [],
        "yapo": [],
        "quintoandar": []
    }

    for t in tasks:
        name = t["name"]

        if "properati" in name:
            groups["properati"].append(t)
        elif "encuentra24" in name:
            groups["encuentra24"].append(t)
        elif "pincali" in name:
            groups["pincali"].append(t)
        elif "icasas" in name:
            groups["icasas"].append(t)
        elif "yapo" in name:
            groups["yapo"].append(t)
        elif "quintoandar" in name:
            groups["quintoandar"].append(t)

    return groups


def run_by_groups(tasks):

    grouped = group_tasks(tasks)

    all_results = []
    all_logs = []

    GROUP_WORKERS = {
        "properati": 3,
        "encuentra24": 2,
        "pincali": 3,
        "icasas": 3,
        "yapo": 2,
        "quintoandar": 2
    }

    for group_name, group_tasks_list in grouped.items():

        if not group_tasks_list:
            continue

        print(f"\n🚀 Ejecutando: {group_name.upper()} ({len(group_tasks_list)} tasks)")

        workers = GROUP_WORKERS.get(group_name, 2)

        tabla, logs = run_scraping_pipeline(
            group_tasks_list,
            max_workers=workers
        )

        if tabla is not None and not tabla.empty:
            all_results.append(tabla)

        if logs is not None and not logs.empty:
            all_logs.append(logs)

        sleep_time = random.uniform(20, 40)
        print(f"⏸️ Esperando {sleep_time:.1f}s...\n")
        time.sleep(sleep_time)

    final_df = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()
    final_logs = pd.concat(all_logs, ignore_index=True) if all_logs else pd.DataFrame()

    return final_df, final_logs

In [ ]:
# Ejecutar
bronze_table, log_df = run_by_groups(tasks)

print("\n✅ RESULTADO")
print(bronze_table.head())

print("\n📊 LOGS")
print(log_df.head())

In [9]:
# =========================================================
# SOLO PINCALI — venta + renta, casas + departamentos
# =========================================================

pincali_tasks = [
    {
        "name": f"pincali_mexico_{op_name}_{inmueble}",
        "func": scrape_pincali,
        "kwargs": {"operation": op, "tipo_inmueble": inmueble, "max_pages": 100},
    }
    for op, op_name in [("venta", "sell"), ("renta", "rent")]
    for inmueble in ["casas", "departamentos"]
]

pincali_bronze, pincali_logs = run_scraping_pipeline(pincali_tasks, max_workers=2)

if not pincali_bronze.empty:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    path = save_csv(pincali_bronze, f"bronze_full_{timestamp}.csv", RAW_DIR)
    print(f"\n💾 Guardado: {path}")
    print("\n✅ RESULTADO PINCALI")
    print(pincali_bronze.groupby(["operation", "type"]).size().to_string())

print("\n📊 LOGS")
print(pincali_logs[["task_name", "status", "rows", "seconds"]].to_string(index=False))

[START] pincali_mexico_sell_casas
[START] pincali_mexico_sell_departamentos


Scraping Pincali - casas - sell: 100%|██████████| 100/100 [06:38<00:00,  3.98s/it]24s/it]


[OK] pincali_mexico_sell_casas | filas=4800
[START] pincali_mexico_rent_casas


Scraping Pincali - departamentos - sell: 100%|██████████| 100/100 [07:15<00:00,  4.35s/it]


[OK] pincali_mexico_sell_departamentos | filas=4800
[START] pincali_mexico_rent_departamentos


Scraping Pincali - casas - rent: 100%|██████████| 100/100 [06:41<00:00,  4.02s/it]60s/it]


[OK] pincali_mexico_rent_casas | filas=4800


Scraping Pincali - departamentos - rent: 100%|██████████| 100/100 [06:34<00:00,  3.95s/it]


[OK] pincali_mexico_rent_departamentos | filas=4800
Bronze data guardada en: scraping_output_round2\raw\bronze_full_20260521_220208.csv

💾 Guardado: scraping_output_round2\raw\bronze_full_20260521_220208.csv

✅ RESULTADO PINCALI
operation  type     
rent       Apartment    4800
           House        4800
sell       Apartment    4800
           House        4800

📊 LOGS
                        task_name  status  rows    seconds
        pincali_mexico_sell_casas success  4800 398.070333
pincali_mexico_sell_departamentos success  4800 435.521672
        pincali_mexico_rent_casas success  4800 401.749003
pincali_mexico_rent_departamentos success  4800 394.808868
